## Export Gold Tables to Parquet

Export all 5 Gold layer tables as single-file Parquet for local consumption by the AI Business Analyst agent, Streamlit dashboard, and any downstream tooling

#### Configuration

In [0]:
CATALOG = "ecommerce"
GOLD_SCHEMA = "gold"
EXPORT_PATH = "/Volumes/ecommerce/gold/raw_data"

GOLD_TABLES = [
    "customer_360",
    "product_performance",
    "inventory_health",
    "fulfillment_metrics",
    "funnel_analytics",
]

print("=" * 60)
print("  GOLD TABLE EXPORT — CONFIGURATION")
print("=" * 60)
print(f"  Catalog:      {CATALOG}")
print(f"  Schema:       {GOLD_SCHEMA}")
print(f"  Export path:  {EXPORT_PATH}")
print(f"  Tables:       {len(GOLD_TABLES)}")
for t in GOLD_TABLES:
    print(f"    • {t}")
print("=" * 60)

  GOLD TABLE EXPORT — CONFIGURATION
  Catalog:      ecommerce
  Schema:       gold
  Export path:  /Volumes/ecommerce/gold/raw_data
  Tables:       5
    • customer_360
    • product_performance
    • inventory_health
    • fulfillment_metrics
    • funnel_analytics


#### Export All Gold Tables

Each table is read from Unity Catalog, coalesced to a single partition, and written as a Parquet file.

In [0]:
import time
import json
from datetime import datetime, timezone

results = []

for table_name in GOLD_TABLES:
    t0 = time.time()
    fqn = f"{CATALOG}.{GOLD_SCHEMA}.{table_name}"
    df = spark.table(fqn)
    row_count = df.count()
    col_count = len(df.columns)
    
    output_path = f"{EXPORT_PATH}/{table_name}.parquet"
    
    (
        df.coalesce(1)
          .write
          .mode("overwrite")
          .parquet(output_path)
    )
    
    elapsed = round(time.time() - t0, 1)
    results.append({
        "table": table_name,
        "rows": row_count,
        "columns": col_count,
        "elapsed": elapsed,
    })
    print(f"  ✓ {table_name:30s}  {row_count:>10,} rows  {col_count:>3} cols  {elapsed}s")

print(f"\n  All {len(GOLD_TABLES)} tables exported.")

# ── Export manifest for provenance ──
manifest = {
    "export_timestamp": datetime.now(timezone.utc).isoformat(),
    "catalog": CATALOG,
    "schema": GOLD_SCHEMA,
    "export_path": EXPORT_PATH,
    "tables": [
        {
            "name": r["table"],
            "rows": r["rows"],
            "columns": r["columns"],
            "export_seconds": r["elapsed"],
        }
        for r in results
    ],
}

manifest_path = f"{EXPORT_PATH}/gold_export_manifest.json"
dbutils.fs.put(manifest_path, json.dumps(manifest, indent=2), overwrite=True)

print(f"\n  Manifest saved: {manifest_path}")

  ✓ customer_360                       100,000 rows   29 cols  13.4s
  ✓ product_performance                 29,120 rows   25 cols  3.2s
  ✓ inventory_health                    29,036 rows   20 cols  4.9s
  ✓ fulfillment_metrics                    903 rows   14 cols  2.8s
  ✓ funnel_analytics                   124,082 rows   17 cols  3.2s

  All 5 tables exported.
Wrote 765 bytes.

  Manifest saved: /Volumes/ecommerce/gold/raw_data/gold_export_manifest.json


#### Verify Exports

In [0]:
import os

print("=" * 60)
print("  EXPORT VERIFICATION")
print("=" * 60)

all_good = True
for table_name in GOLD_TABLES:
    export_dir = f"{EXPORT_PATH}/{table_name}.parquet"
    
    try:
        # Read back the exported Parquet to verify
        verify_df = spark.read.parquet(export_dir)
        verify_count = verify_df.count()
        
        # Compare against source
        source_count = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.{table_name}").count()
        
        match = "✓" if verify_count == source_count else "✗ MISMATCH"
        if verify_count != source_count:
            all_good = False
        
        print(f"  {match}  {table_name:30s}  source={source_count:>10,}  export={verify_count:>10,}")
    except Exception as e:
        all_good = False
        print(f"  ✗  {table_name:30s}  ERROR: {e}")

print(f"\n  {'All exports verified.' if all_good else 'SOME EXPORTS FAILED — check above.'}")
print("=" * 60)

## Next Steps

This notebook exports the Gold tables as point-in-time Parquet snapshots for the local AI platform.

Download the exported `part-*.parquet` files from:

`/Volumes/ecommerce/gold/raw_data/`

Rename them to:

- `customer_360.parquet`
- `product_performance.parquet`
- `inventory_health.parquet`
- `fulfillment_metrics.parquet`
- `funnel_analytics.parquet`

Place them in the project's `data/` directory alongside:

```
data/
├── churn_feature_dataset.parquet
├── predictions.parquet
├── inference_features.parquet
├── customer_360.parquet
├── product_performance.parquet
├── inventory_health.parquet
├── fulfillment_metrics.parquet
└── funnel_analytics.parquet
```

These snapshots are consumed by `07_ai_business_analyst.ipynb`, where they are loaded into DuckDB for business signal generation and AI-powered executive analysis.